In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from tensorflow.keras.datasets import mnist

In [ ]:
(x_train,y_train), (x_test, y_test) = mnist.load_data()

In [ ]:
print("X_train :", x_train.shape)
print("Y_train :", y_train.shape)
print("X_test :", x_test.shape)
print("Y_test :", y_test.shape)

## **Dataset Loading**

**Observation:**  
The MNIST dataset was successfully loaded with **60,000 training images** and **10,000 testing images**.

**Decision:**  
The MNIST dataset was selected for model training and evaluation.

**Reason:**  
It is a standard benchmark dataset for handwritten digit classification.

In [ ]:
plt.imshow(x_train[12], cmap="gray")
plt.show()

## **Image Visualization**

**Observation:**  
A sample handwritten digit image was displayed successfully.

**Decision:**  
The image was visualized before preprocessing.

**Reason:**  
To verify that the dataset was loaded correctly.


In [ ]:
plt.figure(figsize = (20,10))
ax = sns.countplot(x=y_train)

# Add values on top of bars
for container in ax.containers:
    ax.bar_label(container)

plt.title("COUNT OF DIGITS")
plt.xlabel("DIGITS")
plt.ylabel("COUNT")
plt.show()

**Observation:**  
The count plot shows that all digit classes are nearly equally distributed.

**Decision:**  
The dataset was used without applying class balancing.

**Reason:**  
The classes are already balanced.

In [ ]:
x_train = x_train.reshape(x_train.shape[0], -1) / 255.0
x_test = x_test.reshape(x_test.shape[0], -1) / 255.0


## **Data Normalization**

**Observation:**  
Pixel values were converted from **0–255** to **0–1**.

**Decision:**  
The images were normalized before training.

**Reason:**  
Normalization improves model performance and training efficiency.

In [ ]:
print("Flattened train shape:", x_train.shape)
print("Flattened test shape:", x_test.shape)


## **Data Flattening**

**Observation:**  
Each **28 × 28** image was converted into **784 features**.

**Decision:**  
The images were flattened before model training.

**Reason:**  
Machine learning algorithms require one-dimensional feature vectors.

In [ ]:
SAMPLE_SIZE = 10000
rng = np.random.RandomState(42)
sample_idx = rng.choice(x_train.shape[0], SAMPLE_SIZE, replace=False)
X_train = x_train[sample_idx]
y_train_s = y_train[sample_idx]
X_test = x_test
y_test_s = y_test

## **Random Sampling**

**Observation:**  
A random sample of **10,000** training images was selected.

**Decision:**  
The sampled dataset was used for training.

**Reason:**  
It reduces training time while maintaining good performance.


In [ ]:
from tensorflow.keras.datasets import mnist
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
import time

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, n_jobs=-1),
    "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "SVM (RBF)": SVC(kernel='rbf', gamma='scale'),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "MLP (Neural Net)": MLPClassifier(hidden_layer_sizes=(128,64), max_iter=50, random_state=42)
}

results = []
predictions = {}

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train_s)
    train_time = time.time() - start

    start = time.time()
    y_pred = model.predict(X_test)
    predict_time = time.time() - start

    acc = accuracy_score(y_test_s, y_pred)
    predictions[name] = y_pred
    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Train Time (s)": round(train_time, 2),
        "Predict Time (s)": round(predict_time, 2)
    })
    print(f"{name}: accuracy={acc:.4f}, train_time={train_time:.2f}s")

results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
results_df

## **Final Model Selection**

**Observation:**  
**SVM** achieved the highest accuracy among all models.

**Decision:**  
**SVM** was selected as the final model.

**Reason:**  
It provided the best prediction performance on the MNIST dataset.

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=results_df, x="Model", y="Accuracy", palette="viridis")
plt.xticks(rotation=30, ha='right')
plt.title("Model Accuracy Comparison")
plt.ylim(0.8, 1.0)
plt.show()

## **Accuracy Comparison**

**Observation:**  
The bar chart clearly compares the accuracy of all models.

**Decision:**  
A bar chart was used for visualization.

**Reason:**  
It makes model comparison simple and easy to understand.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
y_pred_best = predictions[best_model_name]

print("Best model:", best_model_name)
print(classification_report(y_test_s, y_pred_best))

cm = confusion_matrix(y_test_s, y_pred_best)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix - {best_model_name}")
plt.show()

**Conclusion:**

The MNIST dataset containing 60,000 training images and 10,000 test images was used for handwritten digit classification. The images were normalized, and five machine learning models were trained and compared. The best-performing model achieved the highest accuracy while maintaining good prediction speed. This model can accurately recognize handwritten digits and is suitable for real-world digit classification tasks.

**Challenges:**
1. **Compute time at scale** — training on the full 60,000-image set is very slow for KNN and SVM (KNN compares against every training point at predict time; SVM training cost grows superlinearly with data size). Solved by subsampling training data to 10,000 rows (`SAMPLE_SIZE`) to keep runtime practical for comparison purposes.
2. **Digit confusion** — the confusion matrix shows certain digit pairs (commonly 4/9, 3/5/8, 7/1) get misclassified more than others due to visually similar handwriting shapes; this is a data-level limitation that flattened pixel vectors don't fully resolve, and a CNN (which preserves 2D spatial structure) would likely reduce this further.